<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-30</br>
</div>

</br>

In [ ]:
# TODO 0: 학습된 모델과 토크나이저를 로드해봅시다.

import torch, warnings
warnings.filterwarnings("ignore")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "./results"  # TODO 8(002)에서 저장한 학습 결과 경로

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_path)
print(f"✅ 모델 로드 완료: {model_name} + LoRA 어댑터")
print(f"   어댑터 경로: {adapter_path}")

</br>

# 학습 내용
>이번 장에서는 <strong>결과 확인 및 저장(Result Verification & Model Saving)</strong>에 대해 학습합니다.</br></br>
>파인튜닝된 모델로 추론을 수행하고, 모델과 토크나이저를 학습해봅시다.

</br>

# 추론 (Inference)
> 파인튜닝된 모델에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ChatML 형식의 프롬프트</mark>를 전달하여 응답을 생성합니다.

모델 저장은 매우 중요합니다. LoRA 학습은 데이터 규모와 모델 크기에 따라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수 시간에서 수십 시간</mark>이 소요될 수 있으며, 저장하지 않으면 환경이 초기화될 때 모든 학습 결과가 사라져 **재학습 비용이 그대로 발생**합니다. 또한 Merge(병합)라는 개념도 알아야 합니다. `merge_and_unload()`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LoRA 어댑터(BA)를 원래 가중치(W)에 수식상 합산</mark>하여 단일 모델로 만드는 작업으로, 병합된 모델은 PEFT 라이브러리 없이 일반 `transformers`만으로 로드할 수 있어 **배포 환경에서 의존성을 단순화**할 수 있습니다.

이 내용은 LoRA 학습이 완료된 상태(Ch.5-1_002 참고)를 전제로 하며, 모델 체크포인트, Hugging Face Hub, safetensors 형식 등에 대한 기본 이해가 도움이 됩니다.

In [ ]:
# TODO 1: system/user 메시지로 ChatML 프롬프트를 구성하고, 채팅 템플릿을 적용하여 변환한 뒤 모델로 응답을 생성해봅시다.


</br>

## apply_chat_template 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>tokenize=False</code></td><td>문자열로 반환 (<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토큰화하지 않음</mark>)</td></tr>
    <tr><td style="text-align:center"><code>add_generation_prompt=True</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">assistant 시작 토큰</mark> 추가</td></tr>
    <tr><td style="text-align:center"><code>add_generation_prompt=False</code></td><td>학습 데이터용 (시작 토큰 불필요)</td></tr>
  </tbody>
</table>

</br>

# 모델 저장 (Model Saving)
> 파인튜닝된 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LoRA 어댑터와 토크나이저</mark>를 저장합니다.

In [ ]:
# TODO 2: 모델과 토크나이저를 "./finetuned_model" 경로에 저장하고, 저장된 파일 목록과 크기를 출력해봅시다.


<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파일</th>
      <th style="text-align:center">크기</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">adapter_config.json</td><td style="text-align:center">0.5 KB</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">adapter_model.safetensors</mark></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">3,456.2 KB</mark></td></tr>
    <tr><td style="text-align:center">special_tokens_map.json</td><td style="text-align:center">0.3 KB</td></tr>
    <tr><td style="text-align:center">tokenizer.json</td><td style="text-align:center">1,842.5 KB</td></tr>
    <tr><td style="text-align:center">tokenizer_config.json</td><td style="text-align:center">1.2 KB</td></tr>
  </tbody>
</table>

</br>

## 저장 결과 구조

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파일</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>adapter_model.safetensors</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LoRA 가중치만</mark> 저장 (매우 작음)</td></tr>
    <tr><td style="text-align:center"><code>adapter_config.json</code></td><td>r, alpha, target_modules 등 설정</td></tr>
  </tbody>
</table>

💡LoRA 저장의 효율성
> Full 모델: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수십 GB</mark> / LoRA 어댑터: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">수 MB</mark></br>
> 기본 모델은 공유하고, 어댑터만 교체하여 여러 태스크에 활용할 수 있습니다.

💡저장된 모델을 다시 로드하려면?
> ```python
> from peft import PeftModel
> base_model = AutoModelForCausalLM.from_pretrained("base_model_name")
> model = PeftModel.from_pretrained(base_model, "./finetuned_model")
> ```

</br>

## merge_and_unload — 병합 저장
> LoRA 어댑터를 base model에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">완전히 병합</mark>하여 단일 모델로 저장합니다.

In [ ]:
# TODO 3: merge_and_unload()로 LoRA 어댑터를 base model에 병합하고,


<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">저장 방식</th>
      <th>방법</th>
      <th style="text-align:center">파일 크기</th>
      <th>로드 방법</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어댑터만 저장</mark></td><td><code>save_pretrained</code></td><td style="text-align:center">수 MB</td><td>base model + adapter 로드</td></tr>
    <tr><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">병합 후 저장</mark></td><td><code>merge_and_unload</code> → <code>save_pretrained</code></td><td style="text-align:center">수십 GB</td><td>단일 모델로 로드</td></tr>
  </tbody>
</table>

💡언제 merge_and_unload를 쓰나?
> 추론 시 base model과 adapter를 별도 로드하는 오버헤드를 없애고 싶을 때 사용합니다.</br>
> 배포 환경에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 모델 파일</mark>로 서빙할 때 적합합니다.

💡병합 후 역분리 불가
> `merge_and_unload` 이후에는 어댑터를 분리할 수 없습니다.</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">원본 어댑터를 별도로 보관</mark>한 뒤 병합하세요.

</br>

In [ ]:
# TODO 4: 병합된 모델을 로드하고, 동일한 프롬프트로 추론하여 정상 동작을 확인해봅시다.
